In [4]:
# ==========================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ==========================================================

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import mne
import json

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from braindecode.models import Deep4Net

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ==========================================================
# REPRODUCIBILITY
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ==========================================================
# PATHS
# ==========================================================

EPOCH_DIR = (
    "/Users/prajwalnara/Documents/EEG-motor-imagery/"
    "data/processed/epochs"
)

# ==========================================================
# SAVED SHALLOW CONVNET DATASETS
# ==========================================================

DEEP_CONVNET_SPLIT_DIR = (
    "/Users/prajwalnara/Documents/EEG-motor-imagery/"
    "data/DEEP_convnet_splits"
)

os.makedirs(
    DEEP_CONVNET_SPLIT_DIR,
    exist_ok=True
)

# ==========================================================
# RESULT DIRECTORY
# ==========================================================

DEEP_CONVNET_RESULT_DIR = (
    "/Users/prajwalnara/Documents/EEG-motor-imagery/"
    "results/Deep_convnet_results"
)
os.makedirs(
    DEEP_CONVNET_RESULT_DIR,
    exist_ok=True
)

# ==========================================================
# MODEL DIRECTORY
# ==========================================================
MODEL_DIR = (
    "/Users/prajwalnara/Documents/EEG-motor-imagery/"
    "models/deep_convnet"
)
os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


# ==========================================================
# DATA CONFIGURATION
# ==========================================================

RUNS = ["R04", "R08", "R12"]


# ==========================================================
# TRAINING CONFIGURATION
# ==========================================================

BATCH_SIZE = 16
LEARNING_RATE = 1e-3
NUM_EPOCHS = 100
N_CLASSES = 3
EXPECTED_N_TIMES = 641


# ==========================================================
# DEVICE
# ==========================================================

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")

elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")

else:
    DEVICE = torch.device("cpu")


print("Device:", DEVICE)
print("PyTorch version:", torch.__version__)

Device: mps
PyTorch version: 2.12.1


In [5]:
# ==========================================================
# CELL 2 — SUBJECT EPOCH LOADING FUNCTION
# ==========================================================

def load_subject_epochs(subject_list):
    """
    Load R04, R08 and R12 epochs for each subject.

    A subject is skipped if:
    - any required run file is missing
    - a run has an unexpected number of time points
    - run dimensions are inconsistent

    Returns
    -------
    subject_data : dict
        {
            "S001": {
                "X": (n_epochs, n_channels, n_times),
                "y": (n_epochs,)
            }
        }

    skipped_subjects : list
        Subjects that failed validation.
    """

    subject_data = {}
    skipped_subjects = []

    for subject in subject_list:

        subject_folder = os.path.join(
            EPOCH_DIR,
            subject
        )

        X_runs = []
        y_runs = []

        valid_subject = True


        # ==================================================
        # LOAD EACH RUN
        # ==================================================

        for run in RUNS:

            file_path = os.path.join(
                subject_folder,
                f"{subject}{run}_epo.fif"
            )


            # ----------------------------------------------
            # CHECK 1 — File exists
            # ----------------------------------------------

            if not os.path.exists(file_path):

                print(
                    f"{subject} skipped: "
                    f"missing {run}"
                )

                valid_subject = False
                break


            # ----------------------------------------------
            # Load epochs
            # ----------------------------------------------

            epochs = mne.read_epochs(
                file_path,
                preload=True,
                verbose=False
            )

            X_run = epochs.get_data()
            y_run = epochs.events[:, -1]


            # ----------------------------------------------
            # CHECK 2 — Expected number of time points
            # ----------------------------------------------

            if X_run.shape[2] != EXPECTED_N_TIMES:

                print(
                    f"{subject} skipped: "
                    f"{run} has {X_run.shape[2]} time points "
                    f"(expected {EXPECTED_N_TIMES})"
                )

                valid_subject = False
                break


            X_runs.append(X_run)
            y_runs.append(y_run)


        # ==================================================
        # SKIP INVALID SUBJECT
        # ==================================================

        if not valid_subject:

            skipped_subjects.append(subject)
            continue


        # ==================================================
        # CHECK 3 — RUN DIMENSIONS ARE CONSISTENT
        # ==================================================

        run_shapes = [
            X.shape[1:]
            for X in X_runs
        ]

        if len(set(run_shapes)) != 1:

            print(
                f"{subject} skipped: "
                f"inconsistent run shapes {run_shapes}"
            )

            skipped_subjects.append(subject)
            continue


        # ==================================================
        # COMBINE RUNS
        # ==================================================

        X_subject = np.concatenate(
            X_runs,
            axis=0
        )

        y_subject = np.concatenate(
            y_runs,
            axis=0
        )


        # ==================================================
        # STORE SUBJECT
        # ==================================================

        subject_data[subject] = {
            "X": X_subject,
            "y": y_subject
        }


    return subject_data, skipped_subjects

In [6]:
#LOADING ALL SUBJECTS AND MARKING OUT THE DEFAULTED EPCOHS
# ==========================================================
# CELL 3 — LOAD ALL SUBJECTS
# ==========================================================

ALL_SUBJECTS = [
    f"S{i:03d}"
    for i in range(1, 110)
]


subject_data, skipped_subjects = load_subject_epochs(
    ALL_SUBJECTS
)


print("=" * 60)
print("DATA LOADING SUMMARY")
print("=" * 60)

print("Requested subjects :", len(ALL_SUBJECTS))
print("Loaded subjects    :", len(subject_data))
print("Skipped subjects   :", len(skipped_subjects))

print("\nSkipped subject IDs:")
print(skipped_subjects)


sample_subject = next(iter(subject_data))

X_sample = subject_data[sample_subject]["X"]
y_sample = subject_data[sample_subject]["y"]


print("\nSample subject:", sample_subject)

print("X shape:", X_sample.shape)
print("y shape:", y_sample.shape)

print(
    "Labels:",
    np.unique(
        y_sample,
        return_counts=True
    )
)

S088 skipped: R04 has 513 time points (expected 641)
S092 skipped: R04 has 513 time points (expected 641)
S100 skipped: R04 has 513 time points (expected 641)
DATA LOADING SUMMARY
Requested subjects : 109
Loaded subjects    : 106
Skipped subjects   : 3

Skipped subject IDs:
['S088', 'S092', 'S100']

Sample subject: S001
X shape: (90, 64, 641)
y shape: (90,)
Labels: (array([1, 2, 3], dtype=int32), array([45, 23, 22]))


In [7]:
# ==========================================================
# CELL 4 — EXPERIMENT CONFIGURATIONS
# ==========================================================

EXPERIMENTS = {

    "10 Subjects": {
        "train": [
            f"S{i:03d}"
            for i in range(1, 9)
        ],

        "test": [
            f"S{i:03d}"
            for i in range(9, 11)
        ]
    },


    "20 Subjects": {
        "train": [
            f"S{i:03d}"
            for i in range(1, 18)
        ],

        "test": [
            f"S{i:03d}"
            for i in range(18, 21)
        ]
    },


    "50 Subjects": {
        "train": [
            f"S{i:03d}"
            for i in range(1, 43)
        ],

        "test": [
            f"S{i:03d}"
            for i in range(43, 51)
        ]
    },


    "109 Subjects": {
        "train": [
            f"S{i:03d}"
            for i in range(1, 93)
        ],

        "test": [
            f"S{i:03d}"
            for i in range(93, 110)
        ]
    }
}

In [8]:
# ==========================================================
# CELL 5 — PREPARE EXPERIMENT DATA
# ==========================================================

LABEL_MAP = {
    1: 0,   # Rest
    2: 1,   # Left
    3: 2    # Right
}


def prepare_experiment_data(
    subject_data,
    train_subjects,
    test_subjects,
    val_ratio=0.10,
    seed=42
):

    # ======================================================
    # KEEP ONLY VALID SUBJECTS
    # ======================================================

    train_subjects = [
        subject
        for subject in train_subjects
        if subject in subject_data
    ]

    test_subjects = [
        subject
        for subject in test_subjects
        if subject in subject_data
    ]


    # ======================================================
    # SUBJECT-WISE TRAIN / VALIDATION SPLIT
    # ======================================================

    rng = np.random.default_rng(seed)

    shuffled_train_subjects = train_subjects.copy()

    rng.shuffle(shuffled_train_subjects)


    # At least one subject must be used for validation

    n_val_subjects = max(
        1,
        round(
            len(shuffled_train_subjects)
            * val_ratio
        )
    )


    val_subjects = shuffled_train_subjects[
        :n_val_subjects
    ]

    actual_train_subjects = shuffled_train_subjects[
        n_val_subjects:
    ]


    # ======================================================
    # HELPER FUNCTION TO COMBINE SUBJECT DATA
    # ======================================================

    def combine_subjects(subject_list):

        X = np.concatenate(
            [
                subject_data[subject]["X"]
                for subject in subject_list
            ],
            axis=0
        )

        y = np.concatenate(
            [
                subject_data[subject]["y"]
                for subject in subject_list
            ],
            axis=0
        )


        # ----------------------------------------------
        # Remap labels
        #
        # 1 → 0 : Rest
        # 2 → 1 : Left
        # 3 → 2 : Right
        # ----------------------------------------------

        y = np.array(
            [
                LABEL_MAP[label]
                for label in y
            ],
            dtype=np.int64
        )


        return X, y


    # ======================================================
    # CREATE TRAINING DATA
    # ======================================================

    X_train, y_train = combine_subjects(
        actual_train_subjects
    )


    # ======================================================
    # CREATE VALIDATION DATA
    # ======================================================

    X_val, y_val = combine_subjects(
        val_subjects
    )


    # ======================================================
    # CREATE TEST DATA
    # ======================================================

    X_test, y_test = combine_subjects(
        test_subjects
    )


    # ======================================================
    # STORE TEST SUBJECT ID FOR EACH EPOCH
    # ======================================================

    test_subject_ids = np.concatenate(
        [
            np.repeat(
                subject,
                len(subject_data[subject]["y"])
            )
            for subject in test_subjects
        ]
    )


    # ======================================================
    # PRINT SPLIT SUMMARY
    # ======================================================

    print("\n" + "=" * 60)
    print("SUBJECT SPLIT")
    print("=" * 60)

    print(
        "Training subjects   :",
        len(actual_train_subjects)
    )

    print(
        "Validation subjects :",
        len(val_subjects)
    )

    print(
        "Test subjects       :",
        len(test_subjects)
    )


    print("\nValidation subjects:")
    print(sorted(val_subjects))

    print("\nTest subjects:")
    print(sorted(test_subjects))


    print("\nData shapes:")

    print(
        "X_train:",
        X_train.shape
    )

    print(
        "X_val  :",
        X_val.shape
    )

    print(
        "X_test :",
        X_test.shape
    )


    return (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test,
        test_subject_ids,
        actual_train_subjects,
        val_subjects,
        test_subjects
    )

In [9]:
# ==========================================================
# CELL 6 — NORMALIZE AND SAVE EXPERIMENT DATA
# ==========================================================

def normalize_data(
    X_train,
    X_val,
    X_test
):

    # ------------------------------------------------------
    # Convert to float32 to reduce memory usage
    # ------------------------------------------------------

    X_train = X_train.astype(
        np.float32,
        copy=False
    )

    X_val = X_val.astype(
        np.float32,
        copy=False
    )

    X_test = X_test.astype(
        np.float32,
        copy=False
    )


    # ------------------------------------------------------
    # Calculate statistics from TRAINING DATA ONLY
    #
    # axis 0 → epochs
    # axis 2 → time samples
    #
    # Separate mean/std for each EEG channel
    # ------------------------------------------------------

    channel_mean = X_train.mean(
        axis=(0, 2),
        keepdims=True
    )

    channel_std = X_train.std(
        axis=(0, 2),
        keepdims=True
    )


    # Avoid division by zero

    channel_std[
        channel_std < 1e-8
    ] = 1.0


    # ------------------------------------------------------
    # Apply training statistics to all three sets
    # ------------------------------------------------------

    X_train = (
        X_train - channel_mean
    ) / channel_std

    X_val = (
        X_val - channel_mean
    ) / channel_std

    X_test = (
        X_test - channel_mean
    ) / channel_std


    return (
        X_train,
        X_val,
        X_test,
        channel_mean,
        channel_std
    )

In [10]:
def save_experiment_data(
    experiment_name,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    test_subject_ids,
    channel_mean,
    channel_std
):

    # ------------------------------------------------------
    # Example:
    #
    # "10 Subjects"
    #       ↓
    # "10_subjects"
    # ------------------------------------------------------

    folder_name = (
        experiment_name
        .lower()
        .replace(" ", "_")
    )


    experiment_dir = os.path.join(
        SHALLOW_CONVNET_SPLIT_DIR,
        folder_name
    )


    os.makedirs(
        experiment_dir,
        exist_ok=True
    )


    # ------------------------------------------------------
    # Save data arrays
    # ------------------------------------------------------

    np.save(
        os.path.join(
            experiment_dir,
            "X_train.npy"
        ),
        X_train
    )

    np.save(
        os.path.join(
            experiment_dir,
            "y_train.npy"
        ),
        y_train
    )


    np.save(
        os.path.join(
            experiment_dir,
            "X_val.npy"
        ),
        X_val
    )

    np.save(
        os.path.join(
            experiment_dir,
            "y_val.npy"
        ),
        y_val
    )


    np.save(
        os.path.join(
            experiment_dir,
            "X_test.npy"
        ),
        X_test
    )

    np.save(
        os.path.join(
            experiment_dir,
            "y_test.npy"
        ),
        y_test
    )


    # ------------------------------------------------------
    # Save test subject IDs
    # ------------------------------------------------------

    np.save(
        os.path.join(
            experiment_dir,
            "test_subject_ids.npy"
        ),
        test_subject_ids
    )


    # ------------------------------------------------------
    # Save normalization statistics
    # ------------------------------------------------------

    np.save(
        os.path.join(
            experiment_dir,
            "channel_mean.npy"
        ),
        channel_mean
    )

    np.save(
        os.path.join(
            experiment_dir,
            "channel_std.npy"
        ),
        channel_std
    )


    print(
        f"\nSaved experiment data to:\n"
        f"{experiment_dir}"
    )

In [11]:
# ==========================================================
# CELL 8 — CREATE PYTORCH DATALOADERS
# ==========================================================

def create_dataloaders(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    batch_size=BATCH_SIZE
):

    # ======================================================
    # CONVERT NUMPY ARRAYS TO TENSORS
    # ======================================================

    X_train_tensor = torch.from_numpy(
        X_train
    ).float()

    y_train_tensor = torch.from_numpy(
        y_train
    ).long()


    X_val_tensor = torch.from_numpy(
        X_val
    ).float()

    y_val_tensor = torch.from_numpy(
        y_val
    ).long()


    X_test_tensor = torch.from_numpy(
        X_test
    ).float()

    y_test_tensor = torch.from_numpy(
        y_test
    ).long()


    # ======================================================
    # CREATE DATASETS
    # ======================================================

    train_dataset = TensorDataset(
        X_train_tensor,
        y_train_tensor
    )

    val_dataset = TensorDataset(
        X_val_tensor,
        y_val_tensor
    )

    test_dataset = TensorDataset(
        X_test_tensor,
        y_test_tensor
    )


    # ======================================================
    # CREATE DATALOADERS
    # ======================================================

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )


    return (
        train_loader,
        val_loader,
        test_loader
    )

In [12]:
# ==========================================================
# CELL 9 — CREATE DEEPCONVNET MODEL
# ==========================================================

def create_shallow_convnet(
    n_channels,
    n_times,
    sfreq
):

    model = Deep4Net(
        n_chans=n_channels,
        n_outputs=N_CLASSES,
        n_times=n_times,
        sfreq=sfreq
    )

    model = model.to(DEVICE)

    return model

In [13]:
# ==========================================================
# CELL 10 — TRAIN EEGNET
# ==========================================================

def train_model(
    model,
    train_loader,
    val_loader,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE
):

    # ======================================================
    # LOSS AND OPTIMIZER
    # ======================================================

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )


    # ======================================================
    # TRAINING HISTORY
    # ======================================================

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": []
    }


    # ======================================================
    # BEST MODEL TRACKING
    # ======================================================

    best_val_accuracy = 0.0
    best_epoch = 0
    best_model_state = None


    # ======================================================
    # TRAINING LOOP
    # ======================================================

    for epoch in range(num_epochs):

        # ==================================================
        # TRAINING PHASE
        # ==================================================

        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0


        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)


            # ----------------------------------------------
            # Clear previous gradients
            # ----------------------------------------------

            optimizer.zero_grad()


            # ----------------------------------------------
            # Forward pass
            # ----------------------------------------------

            outputs = model(X_batch)


            # ----------------------------------------------
            # Calculate loss
            # ----------------------------------------------

            loss = criterion(
                outputs,
                y_batch
            )


            # ----------------------------------------------
            # Backpropagation
            # ----------------------------------------------

            loss.backward()


            # ----------------------------------------------
            # Update model weights
            # ----------------------------------------------

            optimizer.step()


            # ----------------------------------------------
            # Accumulate loss
            # ----------------------------------------------

            train_loss += (
                loss.item()
                * X_batch.size(0)
            )


            # ----------------------------------------------
            # Predictions
            # ----------------------------------------------

            predictions = outputs.argmax(
                dim=1
            )


            train_correct += (
                predictions == y_batch
            ).sum().item()

            train_total += y_batch.size(0)


        # ==================================================
        # TRAINING METRICS
        # ==================================================

        train_loss /= train_total

        train_accuracy = (
            train_correct
            / train_total
        )


        # ==================================================
        # VALIDATION PHASE
        # ==================================================

        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0


        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)


                # ------------------------------------------
                # Forward pass
                # ------------------------------------------

                outputs = model(X_batch)


                # ------------------------------------------
                # Validation loss
                # ------------------------------------------

                loss = criterion(
                    outputs,
                    y_batch
                )


                val_loss += (
                    loss.item()
                    * X_batch.size(0)
                )


                # ------------------------------------------
                # Predictions
                # ------------------------------------------

                predictions = outputs.argmax(
                    dim=1
                )


                val_correct += (
                    predictions == y_batch
                ).sum().item()

                val_total += y_batch.size(0)


        # ==================================================
        # VALIDATION METRICS
        # ==================================================

        val_loss /= val_total

        val_accuracy = (
            val_correct
            / val_total
        )


        # ==================================================
        # SAVE HISTORY
        # ==================================================

        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_accuracy"].append(
            val_accuracy
        )


        # ==================================================
        # SAVE BEST MODEL
        # ==================================================

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy
            best_epoch = epoch + 1

            best_model_state = {
                key: value.detach().cpu().clone()
                for key, value
                in model.state_dict().items()
            }


        # ==================================================
        # PRINT PROGRESS
        # ==================================================

        print(
            f"Epoch [{epoch + 1:03d}/{num_epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )


    # ======================================================
    # RESTORE BEST MODEL
    # ======================================================

    model.load_state_dict(
        best_model_state
    )


    print("\n" + "=" * 60)

    print(
        f"Best Validation Accuracy: "
        f"{best_val_accuracy:.4f}"
    )

    print(
        f"Best Epoch: "
        f"{best_epoch}"
    )

    print("=" * 60)


    return (
        model,
        history,
        best_val_accuracy,
        best_epoch
    )

In [14]:
# ==========================================================
# CELL 11 — EVALUATE DEEP CONVNET
# ==========================================================

def evaluate_model(
    model,
    test_loader,
    test_subject_ids
):

    model.eval()


    # ======================================================
    # STORE ALL TRUE LABELS AND PREDICTIONS
    # ======================================================

    all_true = []
    all_predictions = []


    # ======================================================
    # TESTING
    # ======================================================

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)


            # ----------------------------------------------
            # Forward pass
            # ----------------------------------------------

            outputs = model(X_batch)


            # ----------------------------------------------
            # Predicted class
            # ----------------------------------------------

            predictions = outputs.argmax(
                dim=1
            )


            # ----------------------------------------------
            # Move results back to CPU
            # ----------------------------------------------

            all_true.extend(
                y_batch.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )


    # ======================================================
    # CONVERT TO NUMPY ARRAYS
    # ======================================================

    all_true = np.array(
        all_true
    )

    all_predictions = np.array(
        all_predictions
    )


    # ======================================================
    # OVERALL TEST ACCURACY
    # ======================================================

    overall_accuracy = accuracy_score(
        all_true,
        all_predictions
    )


    # ======================================================
    # CLASSIFICATION REPORT
    # ======================================================

    report_text = classification_report(
        all_true,
        all_predictions,
        target_names=[
            "Rest",
            "Left",
            "Right"
        ],
        digits=4,
        zero_division=0
    )


    # ======================================================
    # CONFUSION MATRIX
    # ======================================================

    confusion = confusion_matrix(
        all_true,
        all_predictions
    )


    # ======================================================
    # SUBJECT-WISE ACCURACY
    # ======================================================

    subject_accuracies = {}


    for subject in np.unique(
        test_subject_ids
    ):

        subject_mask = (
            test_subject_ids == subject
        )


        subject_true = all_true[
            subject_mask
        ]

        subject_predictions = all_predictions[
            subject_mask
        ]


        subject_accuracy = accuracy_score(
            subject_true,
            subject_predictions
        )


        subject_accuracies[
            subject
        ] = float(
            subject_accuracy
        )


    # ======================================================
    # PRINT RESULTS
    # ======================================================

    print("\n" + "=" * 60)
    print("TEST RESULTS")
    print("=" * 60)


    print(
        f"\nOverall Accuracy: "
        f"{overall_accuracy:.4f}"
    )


    print(
        "\nClassification Report:\n"
    )

    print(
        report_text
    )


    print(
        "Confusion Matrix:\n"
    )

    print(
        confusion
    )


    print(
        "\nSubject-wise Accuracy:"
    )


    for subject, accuracy in (
        subject_accuracies.items()
    ):

        print(
            f"{subject}: "
            f"{accuracy:.4f}"
        )


    return (
        overall_accuracy,
        report_text,
        confusion,
        subject_accuracies
    )

In [15]:
# ==========================================================
# CELL 12 — SAVE EXPERIMENT RESULTS
# ==========================================================

def save_experiment_results(
    experiment_name,
    overall_accuracy,
    report_text,
    confusion,
    subject_accuracies
):

    # ------------------------------------------------------
    # Example:
    #
    # "10 Subjects"
    #       ↓
    # "10_subjects"
    # ------------------------------------------------------

    folder_name = (
        experiment_name
        .lower()
        .replace(" ", "_")
    )


    experiment_dir = os.path.join(
        SHALLOW_CONVNET_RESULT_DIR,
        folder_name
    )


    os.makedirs(
        experiment_dir,
        exist_ok=True
    )


    # ======================================================
    # 1. SAVE CLASSIFICATION REPORT
    #
    # precision
    # recall
    # F1-score
    # support
    # ======================================================

    report_path = os.path.join(
        experiment_dir,
        "classification_report.txt"
    )


    with open(
        report_path,
        "w"
    ) as file:

        file.write(
            report_text
        )


    # ======================================================
    # 2. SAVE CONFUSION MATRIX
    # ======================================================

    confusion_path = os.path.join(
        experiment_dir,
        "confusion_matrix.npy"
    )


    np.save(
        confusion_path,
        confusion
    )


    # ======================================================
    # 3. SAVE OVERALL + SUBJECT-WISE ACCURACY
    # ======================================================

    accuracy_results = {

        "overall_accuracy": float(
            overall_accuracy
        ),

        "subject_wise_accuracy": {
            subject: float(accuracy)
            for subject, accuracy
            in subject_accuracies.items()
        }
    }


    accuracy_path = os.path.join(
        experiment_dir,
        "accuracy_results.json"
    )


    with open(
        accuracy_path,
        "w"
    ) as file:

        json.dump(
            accuracy_results,
            file,
            indent=4
        )


    # ======================================================
    # PRINT SAVING SUMMARY
    # ======================================================

    print("\n" + "=" * 60)

    print(
        f"Results saved for: "
        f"{experiment_name}"
    )

    print(
        f"Location:\n"
        f"{experiment_dir}"
    )

    print("=" * 60)

In [16]:
# ==========================================================
# CELL 13 — SAVE DEEP CONVNET MODEL
# ==========================================================

def save_model_weights(
    model,
    experiment_name,
    best_epoch,
    best_val_accuracy
):

    folder_name = (
        experiment_name
        .lower()
        .replace(" ", "_")
    )

    experiment_dir = os.path.join(
        MODEL_DIR,
        folder_name
    )

    os.makedirs(
        experiment_dir,
        exist_ok=True
    )

    model_path = os.path.join(
        experiment_dir,
        "best_model.pth"
    )

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "best_epoch": best_epoch,
            "best_val_accuracy": best_val_accuracy,
            "n_channels": N_CHANNELS,
            "n_times": N_TIMES,
            "n_classes": N_CLASSES,
            "sfreq": 160
        },
        model_path
    )

    print(
        f"Model saved to:\n{model_path}"
    )

In [17]:
# ==========================================================
# CELL 14 — RUN REMAINING DEEP CONVNET EXPERIMENTS
# ==========================================================

import gc


experiment_summary = {}


EXPERIMENTS_TO_RUN = [
    "10 Subjects",
    "20 Subjects",
    "50 Subjects",
    "109 Subjects"
]


for EXPERIMENT_NAME in EXPERIMENTS_TO_RUN:

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    print("\n\n")
    print("=" * 70)
    print(f"STARTING EXPERIMENT: {EXPERIMENT_NAME}")
    print("=" * 70)


    # ======================================================
    # 1. GET EXPERIMENT CONFIGURATION
    # ======================================================

    experiment = EXPERIMENTS[
        EXPERIMENT_NAME
    ]


    # ======================================================
    # 2. PREPARE DATA
    # ======================================================

    (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test,
        test_subject_ids,
        train_subjects,
        val_subjects,
        test_subjects

    ) = prepare_experiment_data(

        subject_data,

        experiment["train"],

        experiment["test"]
    )


    # ======================================================
    # 3. NORMALIZE DATA
    # ======================================================

    (
        X_train,
        X_val,
        X_test,
        channel_mean,
        channel_std

    ) = normalize_data(

        X_train,
        X_val,
        X_test
    )


    # ======================================================
    # 4. SAVE PREPARED DATA
    # ======================================================

    save_experiment_data(

        EXPERIMENT_NAME,

        X_train,
        y_train,

        X_val,
        y_val,

        X_test,
        y_test,

        test_subject_ids,

        channel_mean,
        channel_std
    )


    # ======================================================
    # 5. CREATE DATALOADERS
    # ======================================================

    (
        train_loader,
        val_loader,
        test_loader

    ) = create_dataloaders(

        X_train,
        y_train,

        X_val,
        y_val,

        X_test,
        y_test
    )


    # ======================================================
    # 6. CREATE A FRESH EEGNET
    # ======================================================
    N_CHANNELS = X_train.shape[1]
    N_TIMES = X_train.shape[2]
    model = create_shallow_convnet(

        n_channels=N_CHANNELS,

        n_times=N_TIMES,

        sfreq=160
    )


    # ======================================================
    # 7. TRAIN
    # ======================================================

    (
        model,
        history,
        best_val_accuracy,
        best_epoch

    ) = train_model(

        model=model,

        train_loader=train_loader,

        val_loader=val_loader,

        num_epochs=NUM_EPOCHS,

        learning_rate=LEARNING_RATE
    )


    # ======================================================
    # 8. EVALUATE
    # ======================================================

    (
        overall_accuracy,
        report_text,
        confusion,
        subject_accuracies

    ) = evaluate_model(

        model=model,

        test_loader=test_loader,

        test_subject_ids=test_subject_ids
    )


    # ======================================================
    # 9. SAVE RESULTS
    # ======================================================

    save_experiment_results(

        experiment_name=EXPERIMENT_NAME,

        overall_accuracy=overall_accuracy,

        report_text=report_text,

        confusion=confusion,

        subject_accuracies=subject_accuracies
    )


    # ======================================================
    # 10. SAVE BEST MODEL
    # ======================================================

    save_model_weights(

        model=model,

        experiment_name=EXPERIMENT_NAME,

        best_epoch=best_epoch,

        best_val_accuracy=best_val_accuracy
    )


    # ======================================================
    # 11. STORE SMALL SUMMARY IN MEMORY
    # ======================================================

    experiment_summary[
        EXPERIMENT_NAME
    ] = {

        "overall_accuracy": float(
            overall_accuracy
        ),

        "best_val_accuracy": float(
            best_val_accuracy
        ),

        "best_epoch": int(
            best_epoch
        )
    }


    # ======================================================
    # 12. DELETE LARGE OBJECTS
    # ======================================================

    del X_train
    del y_train

    del X_val
    del y_val

    del X_test
    del y_test

    del test_subject_ids

    del channel_mean
    del channel_std

    del train_loader
    del val_loader
    del test_loader

    del model
    del history 


    # ======================================================
    # 13. FORCE MEMORY CLEANUP
    # ======================================================

    gc.collect()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


    print("\n" + "=" * 70)
    print(f"COMPLETED: {EXPERIMENT_NAME}")
    print("=" * 70)




STARTING EXPERIMENT: 10 Subjects

SUBJECT SPLIT
Training subjects   : 7
Validation subjects : 1
Test subjects       : 2

Validation subjects:
['S004']

Test subjects:
['S009', 'S010']

Data shapes:
X_train: (630, 64, 641)
X_val  : (90, 64, 641)
X_test : (180, 64, 641)

Saved experiment data to:
/Users/prajwalnara/Documents/EEG-motor-imagery/data/shallow_convnet_splits/10_subjects
Epoch [001/100] | Train Loss: 1.4214 | Val Loss: 1.0747 | Train Acc: 0.4016 | Val Acc: 0.4000
Epoch [002/100] | Train Loss: 1.3930 | Val Loss: 1.0701 | Train Acc: 0.4397 | Val Acc: 0.4667
Epoch [003/100] | Train Loss: 1.2983 | Val Loss: 0.8991 | Train Acc: 0.4397 | Val Acc: 0.6111
Epoch [004/100] | Train Loss: 1.1735 | Val Loss: 0.9364 | Train Acc: 0.4444 | Val Acc: 0.5444
Epoch [005/100] | Train Loss: 1.1396 | Val Loss: 0.9364 | Train Acc: 0.4524 | Val Acc: 0.5000
Epoch [006/100] | Train Loss: 1.0456 | Val Loss: 0.9197 | Train Acc: 0.4825 | Val Acc: 0.6889
Epoch [007/100] | Train Loss: 1.0813 | Val Loss: 0